In [ ]:
pip install data-designer

In [ ]:
from google.colab import userdata
import os

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

data_designer = DataDesigner()

In [ ]:
config_builder_without_seed = dd.DataDesignerConfigBuilder()

# 1. Sample a category
config_builder_without_seed.add_column(
    dd.SamplerColumnConfig(
        name="language",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["English", "Spanish", "French", "German", "Italian"],
        ),
    )
)

# 2. Generate text using the sampled language
config_builder_without_seed.add_column(
    dd.LLMTextColumnConfig(
        name="greeting",
        model_alias="nvidia-text",
        prompt="Write a casual greeting in {{ language }}.",
    )
)

# 3. Preview
results = data_designer.preview(config_builder_without_seed)
results.display_sample_record()

[07:56:45] [INFO] 🖼️ Preview generation in progress
[07:56:45] [INFO]   |-- 🔒 Jinja rendering engine: secure
[07:56:45] [INFO] ✅ Validation passed
[07:56:45] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[07:56:45] [INFO] 🩺 Running health checks for models...
[07:56:45] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3.5-lightning-30b-a3b' in provider named 'nvidia' for model alias 'nvidia-text'...
[07:56:46] [INFO]   |-- ✅ Passed!
[07:56:46] [INFO] ⚡ Using async task-queue preview
[07:56:46] [INFO] 📝 llm-text model config for column 'greeting'
[07:56:46] [INFO]   |-- model: 'nvidia/nemotron-3.5-lightning-30b-a3b'
[07:56:46] [INFO]   |-- model alias: 'nvidia-text'
[07:56:46] [INFO]   |-- model provider: 'nvidia'
[07:56:46] [INFO]   |-- inference parameters:
[07:56:46] [INFO]   |  |-- generation_type=chat-completion
[07:56:46] [INFO]   |  |-- max_parallel_requests=4
[07:56:46] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[07:56:46] [INFO]   |

[index: 0]                                                  
                                                                                                              
                                              Generated Columns                                               
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name     ┃ Value                                                                                           ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ language │ French                                                                                          │
├──────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ greeting │ Salut ! 😊                                                                                      │
│          │                                                                                                 │
│          │ That's the most common way to say "Hi" or "Hey" in French. It's informal and perfect for        │
│          │ friends or people your age.                                                                     │
└──────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

In [ ]:
config_builder_with_seed = dd.DataDesignerConfigBuilder()

# 1. Register custom model config
config_builder_with_seed.add_model_config(
    dd.ModelConfig(
        alias="nvidia-text-custom",
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        provider="nvidia",
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            max_tokens=2028,
        ),
    )
)

# 2. Attach seed data
config_builder_with_seed.with_seed_dataset(
    dd.LocalFileSeedSource(path="/content/reviews.jsonl")
)

# 3. Add generated column using the custom alias
config_builder_with_seed.add_column(
    dd.LLMTextColumnConfig(
        name="sentiment",
        model_alias="nvidia-text-custom",
        prompt=(
            "Classify this product review as Positive, Negative, "
            "or Neutral. Return only the label.\n"
            "Product: {{ product }}\n"
            "Review: {{ review }}"
        ),
    )
)

config_builder_with_seed.add_column(
    dd.LLMTextColumnConfig(
        name="review_summary",
        model_alias="nvidia-text-custom",
        prompt=(
            "Summarize this product review in one sentence. "
            "Do not invent details.\n"
            "Review: {{ review }}"
        ),
    )
)

from data_designer.config import LLMJudgeColumnConfig, Score

config_builder_with_seed.add_column(
    LLMJudgeColumnConfig(
        name="summary_quality",
        model_alias="nvidia-text-custom",
        prompt="""
Evaluate the summary against the original review.

Original review: {{ review }}
Generated summary: {{ review_summary }}
""",
        scores=[
            Score(
                name="Faithfulness",
                description=(
                    "Judge whether the summary accurately captures "
                    "the important information in the original review "
                    "without inventing facts or omitting critical details."
                ),
                options={
                    1: "Very poor",
                    2: "Poor",
                    3: "Acceptable",
                    4: "Good",
                    5: "Excellent",
                },
            )
        ],
    )
)

# 4. Preview
results = data_designer.preview(config_builder_with_seed)
results.display_sample_record()

[08:25:36] [INFO] 🔁 Preview generation in progress
[08:25:36] [INFO]   |-- 🔒 Jinja rendering engine: secure
[08:25:36] [INFO] ✅ Validation passed
[08:25:36] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[08:25:36] [INFO] 🩺 Running health checks for models...
[08:25:36] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3.5-lightning-30b-a3b' in provider named 'nvidia' for model alias 'nvidia-text-custom'...
[08:25:38] [INFO]   |-- ✅ Passed!
[08:25:38] [INFO] ⚡ Using async task-queue preview
[08:25:38] [INFO] 📝 llm-text model config for column 'sentiment'
[08:25:38] [INFO]   |-- model: 'nvidia/nemotron-3.5-lightning-30b-a3b'
[08:25:38] [INFO]   |-- model alias: 'nvidia-text-custom'
[08:25:38] [INFO]   |-- model provider: 'nvidia'
[08:25:38] [INFO]   |-- inference parameters:
[08:25:38] [INFO]   |  |-- generation_type=chat-completion
[08:25:38] [INFO]   |  |-- max_parallel_requests=4
[08:25:38] [INFO]   |  |-- temperature=1.00
[08:25:38] [INFO]   |  |-- max_tokens=2028
[08:25:38] 

[index: 0]                                                  
                                                                                                              
                                                 Seed Columns                                                 
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name        ┃ Value                                                                                        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product     │ Headphones                                                                                   │
├─────────────┼──────────────────────────────────────────────────────────────────────────────────────────────┤
│ review      │ Amazing sound quality and comfortable fit. Totally worth the price.                          │
└─────────────┴──────────────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                              
                                                                                                              
                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                ┃ Value                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sentiment           │ Positive                                                                             │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ review_summary      │ Amazing sound quality and comfortable fit, totally worth the price.                  │
└─────────────────────┴──────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                              
                                                                                                              
                                       LLM-as-a-Judge: summary_quality                                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Faithfulness                                                                                               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ score: 5                                                                                                   │
│ reasoning: The summary accurately captures all key information from the original review: amazing sound     │
│ quality, comfortable fit, and that it's worth the price. No critical details are omitted or invented.      │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [ ]:
from datetime import datetime
import pandas as pd
import glob
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Run without seed
results_without_seed = data_designer.create(
    config_builder_without_seed,
    num_records=4,
    dataset_name=f"greeting_languages_{timestamp}",
    artifact_path="/content/output",
)

# Run with seed
results_with_seed = data_designer.create(
    config_builder_with_seed,
    num_records=4,
    dataset_name=f"product_reviews_sentiment_{timestamp}",
    artifact_path="/content/output",
)

# Export both runs to JSONL
for results in [results_without_seed, results_with_seed]:
    parquet_dir = str(results.artifact_storage.final_dataset_path)
    files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

    dataset_dir = os.path.dirname(parquet_dir)
    dataset_name = os.path.basename(dataset_dir)
    jsonl_path = os.path.join(dataset_dir, f"{dataset_name}.jsonl")

    df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
    print("Saved:", jsonl_path)

[08:41:19] [INFO] 🎨 Creating Data Designer dataset
[08:41:19] [INFO]   |-- 🔒 Jinja rendering engine: secure
[08:41:19] [INFO] ✅ Validation passed
[08:41:19] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[08:41:19] [INFO] 🩺 Running health checks for models...
[08:41:19] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3.5-lightning-30b-a3b' in provider named 'nvidia' for model alias 'nvidia-text'...
[08:41:20] [INFO]   |-- ✅ Passed!
[08:41:20] [INFO] ⚡ Using async task-queue builder
[08:41:20] [INFO] 📝 llm-text model config for column 'greeting'
[08:41:20] [INFO]   |-- model: 'nvidia/nemotron-3.5-lightning-30b-a3b'
[08:41:20] [INFO]   |-- model alias: 'nvidia-text'
[08:41:20] [INFO]   |-- model provider: 'nvidia'
[08:41:20] [INFO]   |-- inference parameters:
[08:41:20] [INFO]   |  |-- generation_type=chat-completion
[08:41:20] [INFO]   |  |-- max_parallel_requests=4
[08:41:20] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[08:41:20] [INFO]   | 

Saved: /content/output/greeting_languages_20261013_084119/greeting_languages_20261013_084119.jsonl
Saved: /content/output/product_reviews_sentiment_20261013_084119/product_reviews_sentiment_20261013_084119.jsonl
